In [482]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

In [483]:
with open("../data/names.txt", 'r') as f:
    words = f.read().splitlines()

In [484]:
start = ord('a'); end = start + 26
chs = [chr(i) for i in range(start, end)]

stoi = {s:i+1 for i, s in enumerate(chs)}
stoi['.'] = 0

itos = {i:s for s, i in stoi.items()}

In [485]:
#building the data set
X, y = [], []

blocksize = 3
for w in words:
    context = [0] * blocksize

    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        y.append(ix)
        # print(''.join(itos[i] for i in context) + '--->' + itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
y = torch.tensor(y)


In [486]:
X.shape, X.dtype, y.shape, y.type

(torch.Size([228146, 3]),
 torch.int64,
 torch.Size([228146]),
 <function Tensor.type>)

In [487]:
g = torch.Generator().manual_seed(101)


C = torch.randn((27, 2), generator=g)


In [493]:

W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]


In [489]:
sum(p.nelement() for p in parameters)

3481

In [499]:
for p in parameters:
    p.requires_grad = True

In [510]:
for i in range(20000):
    # construct mini batches
    ix = torch.randint(0, X.shape[0], (32,))

    # forward pass
    emb = C[X[ix]] # (inputs, 3, 2)
    Z1 = emb.view((-1, 6)) @ W1 + b1 # (inputs, outputs1 : 100)
    A1 = Z1.tanh() # (inputs, outputs 1 : 100)
    Z2 = A1 @ W2 + b2 # (inputs, outputs2 : 27)
    loss = F.cross_entropy(Z2, y[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data -= 0.01 * p.grad



In [511]:
emb = C[X]
Z1 = emb.view(-1, 6) @ W1 + b1
A1 = Z1.tanh()
Z2 = A1 @ W2 + b2

loss = F.cross_entropy(Z2, y)
loss.item()

2.420910596847534